In [1]:
import sys
from pathlib import Path

# Find project root dynamically
def get_project_root() -> Path:
    try:
        path = Path(__file__).resolve()
        for parent in [path] + list(path.parents):
            if (parent / "requirements.txt").exists() or (parent / "project").exists():
                return parent
    except NameError:
        pass
    path = Path.cwd().resolve()
    for parent in [path] + list(path.parents):
        if (parent / "requirements.txt").exists() or (parent / "project").exists():
            return parent
    return path

ROOT = get_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch
import torch.nn as nn
from torch.nn import functional as F

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# parameters
batch_size = 64          # number of sequences processed together
block_size = 64          # maximum context length
max_iters = 5000         # training iterations
eval_interval = 300
learning_rate = 3e-4

n_embd = 128             # embedding dimension
n_head = 4               # number of attention heads
n_layer = 4              # number of transformer blocks
dropout = 0.2            # dropout probability

# read text file
with open(ROOT / 'TASK 1' / 'input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# create vocabulary
chars = sorted(list(set(text)))
vocab_size = len(chars)

# dictionaries
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}


# encoder-decoder=
def encode(s):
    return [stoi[c] for c in s]

def decode(l):
    return ''.join([itos[i] for i in l])


# encode full dataset
data = torch.tensor(encode(text), dtype=torch.long)

# train - test split
n = int(0.9 * len(data))

train_data = data[:n]
test_data = data[n:]


# batching
# creates training examples for next-token prediction
def get_batch(split):

    data = train_data if split == 'train' else test_data

    # random starting indices
    ix = torch.randint(
        len(data) - block_size,
        (batch_size,)
    )

    # input sequences
    x = torch.stack([
        data[i:i + block_size]
        for i in ix
    ])

    # target sequences
    # shifted by one position
    y = torch.stack([
        data[i + 1:i + block_size + 1]
        for i in ix
    ])

    x = x.to(device)
    y = y.to(device)

    return x, y


# SINGLE ATTENTION HEAD
# one self-attention unit
# learns ONE type of relationship between tokens

class Head(nn.Module):

    def __init__(self, head_size):

        super().__init__()

        # key, query, value projections
        # these are learned linear transformations
        # Query : what is this token looking for?
        # Key   : what information does this token contain?
        # Value : what information should this token pass forward?
        # each token gets transformed into Q,K,V vectors

        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        # causal mask
        # torch.tril creates lower triangular matrix
        # prevents future tokens from being visible

        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        # dropout layer - randomly removes some values during training
        # prevents overfitting
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        # B = batch size
        # T = sequence length
        # C = embedding dimension

        B, T, C = x.shape

        # create key, query, value matrices
        # shape:
        # (B,T,n_embd) -> (B,T,head_size)

        k = self.key(x)       # (B,T,head_size)
        q = self.query(x)     # (B,T,head_size)
        v = self.value(x)     # (B,T,head_size)

        # attention scores
        # every token compares itself with every other token
        # transpose(-2,-1): swaps last two dimensions
        # @ = matrix multiplication
        # shape: (B,T,head_size) @ (B,head_size,T) -> (B,T,T)
        wei = q @ k.transpose(-2, -1)

        # scale scores
        # prevents values from becoming too large
        # stabilizes softmax and gradients
        wei = wei * (k.size(-1) ** -0.5)

        # apply causal mask
        # masked_fill(condition, value) -
        # wherever condition is True, replace with -inf
        # after softmax: softmax(-inf) = 0
        # so future tokens receive zero attention

        wei = wei.masked_fill(
            self.tril[:T, :T] == 0,
            float('-inf')
        )

        # convert scores to probabilities
        # softmax along last dimension
        # each row now sums to 1
        wei = F.softmax(wei, dim=-1)

        # dropout on attention probabilities
        wei = self.dropout(wei)

        # weighted aggregation of values
        # attention weights multiply value vectors
        # tokens gather information from important tokens
        # shape: (B,T,T) @ (B,T,head_size) -> (B,T,head_size)
        out = wei @ v

        return out

# BLOCK 1
# MULTI HEAD ATTENTION
# instead of using ONE attention head, transformers use MANY heads in parallel
# different heads learn different relationships
# examples:
# - one head learns punctuation
# - one learns nearby context
# - one learns long-range dependencies
# all heads see same input
# but each head has separate Q,K,V matrices

class MultiHeadAttention(nn.Module):

    def __init__(self, num_heads, head_size):

        super().__init__()

        # ModuleList:
        # special PyTorch list for storing trainable modules
        # each Head is independent

        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])

        # projection layer - learns how to combine information from all heads
        # concatenated heads are mixed together here
        # without this layer: heads would remain independent

        self.proj = nn.Linear( num_heads * head_size, n_embd)

        # dropout layer
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        # concatenate outputs from all heads
        out = torch.cat([h(x) for h in self.heads], dim=-1)

        # projection layer
        # mixes information from all heads
        out = self.proj(out)

        # dropout on final output features
        out = self.dropout(out)

        return out


# BLOCK 2
# FEED FORWARD NETWORK (MLP)
# attention mixes information BETWEEN tokens
# feed-forward network processes EACH token independently
# helps model learn:
# - higher level features
# - nonlinear transformations

# structure:
# Linear -> ReLU -> Linear

class FeedForward(nn.Module):

    def __init__(self, n_embd):

        super().__init__()

        # Sequential:
        # runs layers one after another automatically
        self.net = nn.Sequential(

            # expand dimension
            # transformer MLPs are usually 4x wider
            nn.Linear( n_embd, 4 * n_embd),

            # non-linearity
            # without activation functions, entire network becomes only linear algebra
            nn.ReLU(),

            # project back to original dimension
            nn.Linear(4 * n_embd, n_embd),

            # dropout
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)


# TRANSFORMER BLOCK
# 1. multi-head attention
# 2. feed-forward network
# 3. layer normalization
# 4. residual connections

class Block(nn.Module):

    def __init__(self, n_embd, n_head):

        super().__init__()

        # dimensions handled by each head
        # example:
        # 128 embedding dim / 4 heads = 32 per head

        head_size = n_embd // n_head

        # multi-head self-attention
        self.sa = MultiHeadAttention(n_head, head_size)

        # feed-forward network
        self.ffwd = FeedForward(n_embd)

        # layer normalization
        # normalizes activations
        # stabilizes training
        # applied across embedding dimension

        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):

        # PRE-NORM TRANSFORMER
        # normalization happens before sublayer
        # modern transformers mostly use pre-norm
        # because training is more stable

        # residual connection
        # x + sublayer(x)
        # model learns "small correction to x" instead of complete transformation

        x = x + self.sa(self.ln1(x))

        # second residual connection
        x = x + self.ffwd(self.ln2(x))

        return x


# TRANSFORMER MODEL

class TransformerLanguageModel(nn.Module):

    def __init__(self):

        super().__init__()

        # token embeddings
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)

        # positional embeddings
        self.position_embedding_table = nn.Embedding(block_size, n_embd)

        # stack multiple transformer blocks
        self.blocks = nn.Sequential(

            Block(n_embd, n_head),
            Block(n_embd, n_head),
            Block(n_embd, n_head),
            Block(n_embd, n_head),

            # final layer normalization
            nn.LayerNorm(n_embd)
        )

        # final linear layer
        # converts embeddings -> vocabulary logits
        # each token position predicts: probability distribution over vocabulary

        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):

        B, T = idx.shape

        # token embeddings
        tok_emb = self.token_embedding_table(idx)
        # (B,T,n_embd)

        # positional embeddings
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        # (T,n_embd)

        x = tok_emb + pos_emb

        # pass through transformer blocks
        x = self.blocks(x)

        # final logits
        logits = self.lm_head(x)
        # (B,T,vocab_size)

        # calculate loss
        if targets is None:

            loss = None

        else:

            B, T, C = logits.shape

            # flatten logits
            # cross entropy expects:
            # logits -> (N,C)

            logits = logits.view(B * T, C)

            # flatten targets
            # cross entropy expects:
            # targets -> (N)

            targets = targets.view(B * T)

            # cross entropy loss
            loss = F.cross_entropy(logits, targets)

        return logits, loss


    def generate(self, idx, max_new_tokens):

        for _ in range(max_new_tokens):

            idx_cond = idx[:, -block_size:]

            # get predictions
            logits, loss = self(idx_cond)

            logits = logits[:, -1, :]
            # (B,vocab_size)

            # convert logits -> probabilities
            probs = F.softmax(logits, dim=-1)

            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)

        return idx


# model
model = TransformerLanguageModel().to(device)
print(sum(p.numel() for p in model.parameters()) / 1e6, 'M parameters')


# optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)


# training loop
for iter in range(max_iters):

    # get training batch
    xb, yb = get_batch('train')

    # forward pass
    logits, loss = model(xb, yb)

    # clear old gradients
    optimizer.zero_grad(set_to_none=True)

    loss.backward()

    # update weights
    optimizer.step()

    # print loss
    if iter % eval_interval == 0:
        print(f"step {iter}: loss {loss.item():.4f}")


# initial context token
context = torch.zeros((1, 1), dtype=torch.long,device=device)

# autoregressive text generation
# repeatedly:
# 1. predict next token
# 2. append token
# 3. feed back into model

generated_tokens = model.generate(context, max_new_tokens=500)

generated_text = decode(generated_tokens[0].tolist())

print("\nGenerated Text:\n")
print(generated_text)


FileNotFoundError: [Errno 2] No such file or directory: 'input.txt'